In [ ]:
# ==============================================================================
# KROK 1: Generowanie REALISTYCZNYCH 4 grup klientów z jawnymi centrami
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
np.random.seed(42)

# Definiujemy 4 realne centra grup w skali rynkowej: [Dochód w tys. PLN, Wydatki 1-100]
# 1. Niski dochód, Mało wydają (Oszczędni)
# 2. Niski dochód, Dużo wydają (Aspirujący / Młodzi)
# 3. Wysoki dochód, Mało wydają (Potencjał)
# 4. Wysoki dochód, Dużo wydają (VIP)
centra = np.array([
    [35, 25],
    [40, 80],
    [120, 20],
    [140, 85]
])

# Generujemy po 100 klientów wokół każdego centrum
dane_list = []
for c in centra:
    dochody = np.random.normal(c[0], 8, 100).clip(20, 200)       # Dochód nigdy nie spadnie poniżej 20k
    wydatki = np.random.normal(c[1], 7, 100).clip(1, 100)        # Wydatki w skali 1-100
    dane_list.append(np.column_stack([dochody, wydatki]))

X_raw = np.vstack(dane_list)
df_klienci = pd.DataFrame(X_raw, columns=['Roczny_Dochod_kPLN', 'Wskaznik_Wydatkow_1_100']).round(1)

# Skalowanie (niezbędne dla k-Means)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_klienci)

plt.scatter(df_klienci['Roczny_Dochod_kPLN'], df_klienci['Wskaznik_Wydatkow_1_100'], color='gray', alpha=0.7, s=40)
plt.title("Surowe dane klientów (Realistyczne dochody od 20k do 160k PLN)", fontsize=13)
plt.xlabel("Roczny dochód (tys. PLN)")
plt.ylabel("Wskaźnik wydatków w sklepie (1-100)")
plt.show()

In [ ]:
# ==============================================================================
# KROK 2: Dobór optymalnego K: Wykres Łokcia (Inertia) oraz Silhouette Score
# ==============================================================================
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

K_zakres = range(2, 9)
inertia_lista = []
silhouette_lista = []

for k in K_zakres:
    km = KMeans(n_clusters=k, n_init='auto', random_state=42)
    km.fit(X_scaled)
    inertia_lista.append(km.inertia_)
    silhouette_lista.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Wykres Metody Łokcia
axes[0].plot(K_zakres, inertia_lista, 'bo-', lw=2)
axes[0].set_title("1. Metoda Łokcia (Szukamy zagięcia)", fontsize=12)
axes[0].set_xlabel("Liczba klastrów (K)")
axes[0].set_ylabel("Inercja (Suma kwadratów odległości)")

# Wykres Silhouette Score (Im wyższy, tym lepsza spójność)
axes[1].plot(K_zakres, silhouette_lista, 'ro-', lw=2)
axes[1].set_title("2. Silhouette Score (Im wyżej, tym lepiej!)", fontsize=12)
axes[1].set_xlabel("Liczba klastrów (K)")
axes[1].set_ylabel("Współczynnik Sylwetki")

plt.tight_layout()
plt.show()

optymalne_k = K_zakres[np.argmax(silhouette_lista)]
print(f"🎯 Wynik matematyczny: Najwyższy Silhouette Score ({max(silhouette_lista):.3f}) wskazuje K = {optymalne_k}!")

In [ ]:
# ==============================================================================
# KROK 3: Finalny model k-Means i inteligentne profilowanie biznesowe
# ==============================================================================
kmeans_final = KMeans(n_clusters=4, n_init='auto', random_state=42)
df_klienci['Segment'] = kmeans_final.fit_predict(X_scaled)

# Centroidy w oryginalnej skali
centroidy = scaler.inverse_transform(kmeans_final.cluster_centers_)

plt.figure(figsize=(10, 6))
paleta = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
sns.scatterplot(data=df_klienci, x='Roczny_Dochod_kPLN', y='Wskaznik_Wydatkow_1_100', 
                hue='Segment', palette=paleta, s=60, alpha=0.8)

# Rysujemy centra grup
plt.scatter(centroidy[:, 0], centroidy[:, 1], s=200, c='black', marker='X', label='Centroidy (Środki grup)')
plt.title("Finalna Segmentacja Klientów (k-Means, K=4)", fontsize=14)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

# Profilowanie statystyczne
profil = df_klienci.groupby('Segment').mean().round(1)

# Inteligentne reguły nadawania nazw biznesowych na podstawie RZECZYWISTYCH średnich
srednia_krajowa_dochod = df_klienci['Roczny_Dochod_kPLN'].mean()
srednie_wydatki = df_klienci['Wskaznik_Wydatkow_1_100'].mean()

def nazwij_segment(row):
    wysoki_dochod = row['Roczny_Dochod_kPLN'] > srednia_krajowa_dochod
    duze_wydatki = row['Wskaznik_Wydatkow_1_100'] > srednie_wydatki
    
    if wysoki_dochod and duze_wydatki:
        return "Wysoki dochód / Dużo wydają (VIP)"
    elif wysoki_dochod and not duze_wydatki:
        return "Wysoki dochód / Mało wydają (Potencjał)"
    elif not wysoki_dochod and duze_wydatki:
        return "Niski dochód / Dużo wydają (Młodzi/Aspirujący)"
    else:
        return "Niski dochód / Mało wydają (Oszczędni)"

profil['Nazwa_Biznesowa'] = profil.apply(nazwij_segment, axis=1)

print("--- PROFIL BIZNESOWY WYODRĘBNIONYCH SEGMENTÓW ---")
display(profil)

In [ ]:
# ==============================================================================
# KROK 4: Dlaczego k-Means polegnie na nieliniowości i jak ratuje nas DBSCAN?
# ==============================================================================
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN

# Tworzymy problem nieliniowy + dodajemy losowy szum (Anomalie / Oszustwa)
X_moons, _ = make_moons(n_samples=300, noise=0.08, random_state=42)
outliers = np.random.uniform(low=-1.5, high=2.5, size=(25, 2)) # 25 podejrzanych transakcji
X_moons_dirty = np.vstack([X_moons, outliers])

# 1. k-Means (K=2)
km_moons = KMeans(n_clusters=2, n_init='auto', random_state=42).fit(X_moons_dirty)

# 2. DBSCAN (gęstościowy)
dbscan = DBSCAN(eps=0.22, min_samples=6).fit(X_moons_dirty)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Wykres k-Means
axes[0].scatter(X_moons_dirty[:, 0], X_moons_dirty[:, 1], c=km_moons.labels_, cmap='coolwarm', s=40)
axes[0].set_title("Porażka k-Means: Tnie przestrzeń sferycznie na pół\n(Nie radzi sobie z nieliniowością i szumem)", fontsize=11)

# Wykres DBSCAN
labels_db = dbscan.labels_
axes[1].scatter(X_moons_dirty[labels_db != -1, 0], X_moons_dirty[labels_db != -1, 1], 
                c=labels_db[labels_db != -1], cmap='viridis', s=40, label='Zidentyfikowane Klastry')
axes[1].scatter(X_moons_dirty[labels_db == -1, 0], X_moons_dirty[labels_db == -1, 1], 
                color='red', marker='x', s=80, label='ANOMALIE / SZUM (-1)')
axes[1].set_title("Triumf DBSCAN: Idealnie odnalazł nieliniowe kształty\noraz automatycznie wykrył szum/oszustwa (czerwone X)", fontsize=11)
axes[1].legend()

plt.tight_layout()
plt.show()